# 03 - Foundry IQ: grounding the agent in policy documents

**Maple Mortgage Assistant, Northwind Home Lending (fictional)**

## What you'll learn
- Chunk and index policy documents in **Azure AI Search** (vector + keyword + semantic ranker)
- Baseline retrieval: a plain hybrid query
- **Foundry IQ**: a *knowledge source* + *knowledge base* that plans queries, retrieves and returns citations
- **Register everything with Foundry, in the notebook**: project index, project connection, knowledge source, knowledge base, agent version with an MCP tool, then list/get them back and clean up
- Build an **Agent Framework** agent (client, `Agent(...)`, context provider, session, run and stream) on the same knowledge base
- When to use classic RAG vs agentic retrieval

### Where each asset lives
| Asset | Created with | Lives in | Portal |
|---|---|---|---|
| Search index, knowledge source, knowledge base | `azure-search-documents` (`SearchIndexClient`) | Azure AI Search service | Azure portal > AI Search > Agentic retrieval |
| Project index (pointer) | `AIProjectClient.indexes` | Foundry project | Foundry portal > Build > Knowledge (Indexes) |
| Project connection (`RemoteTool`) | ARM REST (the SDK can only read connections) | Foundry project | Foundry portal > Management center > Connected resources |
| Agent + version with an `MCPTool` | `AIProjectClient.agents.create_version` | Foundry project | Foundry portal > Build > Agents |
| Agent Framework `Agent` (section 8) | `agent_framework` | **Not registered**: it is code in your process | n/a |

> **[preview]** Knowledge bases / knowledge sources and the KB MCP endpoint are preview (`2026-08-01-preview`). Cells that need them print the real error if your region/tenant lacks the capability; nothing is simulated.

In [1]:
import os, sys, asyncio, json
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

DAY2 = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(DAY2 / ".env", override=False)
sys.path.insert(0, str(DAY2))

REQUIRED = ["AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_AI_PROJECT_ENDPOINT", "AZURE_AI_MODEL_DEPLOYMENT_NAME", "AZURE_AI_EMBEDDING_DEPLOYMENT_NAME", "AZURE_SEARCH_ENDPOINT", "AZURE_SEARCH_INDEX_NAME", "AZURE_SEARCH_KNOWLEDGE_BASE_NAME", "AZURE_AI_KB_CONNECTION_NAME"]
for name in REQUIRED:
    print(f"{name:42s}", "set" if os.environ.get(name) else "MISSING")   # never print values
assert all(os.environ.get(n) for n in REQUIRED), "Run scripts/write-env.sh first"

credential = DefaultAzureCredential()

import re, time
from urllib.parse import urlparse
from azure.ai.projects import AIProjectClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
CHAT, EMBED = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"], os.environ["AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"]
SEARCH = os.environ["AZURE_SEARCH_ENDPOINT"]
INDEX, KB = os.environ["AZURE_SEARCH_INDEX_NAME"], os.environ["AZURE_SEARCH_KNOWLEDGE_BASE_NAME"]
KB_CONN = os.environ["AZURE_AI_KB_CONNECTION_NAME"]
KS = f"{INDEX}-source"                                     # knowledge source name
OPENAI_URL = f"https://{urlparse(ENDPOINT).hostname.split('.')[0]}.openai.azure.com"   # Azure OpenAI endpoint of the Foundry account

pc = AIProjectClient(endpoint=ENDPOINT, credential=credential, allow_preview=True)
from azure.identity import get_bearer_token_provider
from openai import AzureOpenAI
aoai = AzureOpenAI(azure_endpoint=OPENAI_URL, api_version="2024-10-21",
                   azure_ad_token_provider=get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default"))
oai = pc.get_openai_client()      # project client: used for agents (Responses API)
index_client = SearchIndexClient(SEARCH, credential)
# Management-plane coordinates (needed to create project connections: the SDK can only read them)
import requests
SUB, RG = os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"]
ACCOUNT = urlparse(ENDPOINT).hostname.split(".")[0]
PROJECT = urlparse(ENDPOINT).path.rstrip("/").split("/")[-1]
ARM_CONN = (f"https://management.azure.com/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.CognitiveServices"
            f"/accounts/{ACCOUNT}/projects/{PROJECT}/connections")
print("search index:", INDEX, "| knowledge base:", KB, "| project:", PROJECT)

AZURE_SUBSCRIPTION_ID                      set
AZURE_RESOURCE_GROUP                       set
AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set
AZURE_SEARCH_ENDPOINT                      set
AZURE_SEARCH_INDEX_NAME                    set
AZURE_SEARCH_KNOWLEDGE_BASE_NAME           set
AZURE_AI_KB_CONNECTION_NAME                set


search index: northwind-policies | knowledge base: northwind-kb | project: maple-mortgage


## 0. What is already registered in the project?
We will register several assets. This helper lists what the **Foundry project** and the **Search service** know about, so you can compare before and after (section 9).

In [2]:
def ledger():
    """Print the assets this notebook registers, read back from the services."""
    print("project connections :", sorted(c.name for c in pc.connections.list()))
    print("project indexes     :", sorted((i.name, i.version) for i in pc.indexes.list()))
    print("agents              :", sorted(a.name for a in pc.agents.list()))
    print("search indexes      :", sorted(i.name for i in index_client.list_indexes()))
    try:
        print("knowledge sources   :", sorted(k.name for k in index_client.list_knowledge_sources()))
        print("knowledge bases     :", sorted(k.name for k in index_client.list_knowledge_bases()))
    except Exception as exc:
        print("knowledge sources/bases: not listable here:", type(exc).__name__, str(exc)[:120])
ledger()

project connections : ['appinsights', 'maple-kb-mcp-nb03', 'northwind-kb-mcp', 'search']


project indexes     : []


agents              : ['maple-hosted', 'maple-learn-mcp', 'maple-policy-iq', 'maple-prompt', 'nb02-maple-fn', 'nb02-maple-learn-mcp']


search indexes      : ['northwind-policies']
knowledge sources   : ['northwind-policies-source']


knowledge bases     : ['northwind-kb']


## 1. Chunk the policy documents
One chunk per `##` section, each prefixed with the policy id and title so the chunk stands alone. Small, self-contained chunks give precise citations.

In [3]:
POLICY_DIR = DAY2 / "data" / "policies"
chunks = []
for f in sorted(POLICY_DIR.glob("*.md")):
    text = f.read_text(encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ").strip()
    policy_id = title.split()[0]
    for n, part in enumerate(re.split(r"\n(?=## )", text)[1:], start=1):
        heading = part.splitlines()[0].lstrip("# ").strip()
        chunks.append({"id": f"{f.stem}-{n}", "policy_id": policy_id, "title": title, "section": heading,
                       "source": f.name, "content": f"{title} / {heading}\n" + "\n".join(part.splitlines()[1:]).strip()})
print(len(chunks), "chunks from", len(list(POLICY_DIR.glob('*.md'))), "files")
print(chunks[3]["id"], "|", chunks[3]["section"], "|", len(chunks[3]["content"]), "chars")

40 chunks from 8 files
01-eligibility-4 | 4. Credit score bands | 347 chars


## 2. Create the index
Vector field (HNSW, 1536 dims for `text-embedding-3-small`), a vectorizer so queries embed themselves (the Search service's managed identity calls the embedding deployment; no keys), and a semantic configuration.

In [4]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType, VectorSearch,
    HnswAlgorithmConfiguration, VectorSearchProfile, AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch)

index = SearchIndex(
    name=INDEX,
    fields=[
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SimpleField(name="policy_id", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchableField(name="title", type=SearchFieldDataType.String),
        SearchableField(name="section", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SearchField(name="content_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                    searchable=True, vector_search_dimensions=1536, vector_search_profile_name="hnsw-profile"),
    ],
    vector_search=VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
        profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw", vectorizer_name="aoai")],
        vectorizers=[AzureOpenAIVectorizer(vectorizer_name="aoai", parameters=AzureOpenAIVectorizerParameters(
            resource_url=OPENAI_URL, deployment_name=EMBED, model_name="text-embedding-3-small"))],
    ),
    semantic_search=SemanticSearch(default_configuration_name="semantic", configurations=[SemanticConfiguration(
        name="semantic", prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"), content_fields=[SemanticField(field_name="content")]))]),
)
index_client.create_or_update_index(index)
print("index ready:", INDEX)

index ready: northwind-policies


### Register the index with the Foundry project
Azure AI Search owns the data. Foundry keeps a small **index asset** (a pointer: connection name + index name) so agents, evaluations and the portal can reference it. It uses the project connection `search` (created by Bicep, authenticates with the project's managed identity).

In [5]:
from azure.ai.projects.models import AzureAISearchIndex

proj_index = pc.indexes.create_or_update(
    name="maple-policies", version="1",
    index=AzureAISearchIndex(connection_name="search", index_name=INDEX, description="Northwind policy chunks (fictional)"))
print("project index:", proj_index.name, "v" + proj_index.version, "->", proj_index.as_dict().get("indexName"))
got = pc.indexes.get(name="maple-policies", version="1")       # read it back
print("get back     :", got.name, got.as_dict().get("connectionName"), got.as_dict().get("indexName"))

project index: maple-policies v1 -> northwind-policies
get back     : maple-policies search northwind-policies


## 3. Embed and upload
Embeddings come from the project's OpenAI-compatible client (Entra auth).

In [6]:
vectors = []
for i in range(0, len(chunks), 16):
    batch = [c["content"] for c in chunks[i:i + 16]]
    vectors += [d.embedding for d in aoai.embeddings.create(model=EMBED, input=batch).data]
docs = [{**c, "content_vector": v} for c, v in zip(chunks, vectors)]

search = SearchClient(SEARCH, INDEX, credential)
result = search.upload_documents(docs)
print("uploaded:", sum(r.succeeded for r in result), "of", len(docs))
for _ in range(20):                      # indexing is near-real-time
    if search.get_document_count() == len(docs):
        break
    time.sleep(2)
print("documents in index:", search.get_document_count())

uploaded: 40 of 40


documents in index: 40


## 4. Baseline: hybrid + semantic query
This is classic RAG retrieval: you write the query, you format the results.

In [7]:
from azure.search.documents.models import VectorizableTextQuery
q = "What is the minimum down payment on a 600k home?"
hits = search.search(search_text=q, vector_queries=[VectorizableTextQuery(text=q, k_nearest_neighbors=5, fields="content_vector")],
                     query_type="semantic", semantic_configuration_name="semantic", top=3,
                     select=["id", "policy_id", "section", "content"])
for h in hits:
    print(f"{h['@search.reranker_score']:.2f}  {h['policy_id']} / {h['section']}")

2.37  NHL-POL-101 / 3. Down payment
2.08  NHL-POL-101 / 2. Property types
1.56  NHL-POL-103 / 1. All applicants


## 5. Foundry IQ: knowledge source + knowledge base [preview]
- **Knowledge source** = pointer to the index (which fields to return, which semantic config).
- **Knowledge base** = the reasoning layer: an LLM plans sub-queries from the chat, runs them against the sources, and returns grounded results with references.

In [8]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters, KnowledgeBase,
    KnowledgeSourceReference, KnowledgeBaseAzureOpenAIModel, AzureOpenAIVectorizerParameters, SearchIndexFieldReference)

ks = SearchIndexKnowledgeSource(
    name=KS, description="Northwind underwriting and compliance policies",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX, semantic_configuration_name="semantic",
        source_data_fields=[SearchIndexFieldReference(name=n) for n in ("id", "policy_id", "section", "source", "content")]))
index_client.create_or_update_knowledge_source(ks)

kb = KnowledgeBase(
    name=KB, description="Maple policy knowledge base",
    knowledge_sources=[KnowledgeSourceReference(name=KS)],
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
        resource_url=OPENAI_URL, deployment_name=CHAT, model_name=CHAT))],
    retrieval_instructions="Northwind mortgage policies. Cite the policy id and section.",
    answer_instructions="Answer only from the retrieved passages. If they do not contain the answer say you do not know.",
)
index_client.create_or_update_knowledge_base(kb)
print("knowledge source:", KS, "| knowledge base:", KB)

knowledge source: northwind-policies-source | knowledge base: northwind-kb


In [9]:
# Read the knowledge source and knowledge base back from the Search service
ks_back = index_client.get_knowledge_source(KS)
kb_back = index_client.get_knowledge_base(KB)
print("knowledge source:", ks_back.name, "->", ks_back.search_index_parameters.search_index_name)
print("knowledge base  :", kb_back.name, "sources:", [s.name for s in kb_back.knowledge_sources])

knowledge source: northwind-policies-source -> northwind-policies
knowledge base  : northwind-kb sources: ['northwind-policies-source']


### Query the knowledge base directly
`KnowledgeBaseRetrievalClient.retrieve` returns passages plus references (what the MCP tool uses under the hood).

In [10]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import KnowledgeBaseRetrievalRequest, KnowledgeRetrievalSemanticIntent

kb_client = KnowledgeBaseRetrievalClient(SEARCH, credential, knowledge_base_name=KB)
resp = kb_client.retrieve(KnowledgeBaseRetrievalRequest(
    intents=[KnowledgeRetrievalSemanticIntent(search="maximum total debt service ratio and gift funds down payment")],
    include_activity=True))
print("references:", len(resp.references or []))
for r in resp.references or []:
    d = r.as_dict()
    print(f" - {d.get('docKey'):18s} {d.get('title')}  (reranker {float(d.get('rerankerScore') or 0):.2f})")
print("activity steps:", [a.as_dict().get("type") for a in resp.activity or []])

references: 5
 - 02-dti-limits-2    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.42)
 - 02-dti-limits-1    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.41)
 - 02-dti-limits-3    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.30)
 - 02-dti-limits-4    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.27)
 - 02-dti-limits-5    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 1.94)
activity steps: ['searchIndex', 'agenticReasoning']


## 6. Register a project connection for the KB MCP endpoint
A Foundry agent calls an MCP server through a **project connection**: `category: RemoteTool`, `authType: ProjectManagedIdentity`, `audience: https://search.azure.com/`, target = the knowledge base MCP URL.
The Bicep deployment already created `northwind-kb-mcp`; here we create our own, `maple-kb-mcp-nb03`, with an ARM `PUT` so you can see every property.
> The Python SDK can **list/get** connections but not create them, so this is the one asset created through the Azure management REST API (`api-version=2025-10-01-preview`).
> The role the project's managed identity needs on Search is **Search Index Data Reader** (granted by Bicep).

In [11]:
MCP_URL = f"{SEARCH}/knowledgebases/{KB}/mcp?api-version=2026-08-01-preview"
NB_CONN = "maple-kb-mcp-nb03"
arm_headers = {"Authorization": "Bearer " + credential.get_token("https://management.azure.com/.default").token,
               "Content-Type": "application/json"}          # token is never printed
body = {"properties": {"category": "RemoteTool", "authType": "ProjectManagedIdentity", "target": MCP_URL,
                       "audience": "https://search.azure.com/", "isSharedToAll": False, "metadata": {"ApiType": "Azure"}}}
r = requests.put(f"{ARM_CONN}/{NB_CONN}?api-version=2025-10-01-preview", headers=arm_headers, json=body, timeout=60)
r.raise_for_status()
print("created connection:", r.json()["name"], "| category:", r.json()["properties"]["category"], "| auth:", r.json()["properties"]["authType"])
conn = pc.connections.get(NB_CONN)                           # SDK read-back
print("SDK get           :", conn.name, conn.type, "| target ok:", conn.target.startswith(SEARCH))

created connection: maple-kb-mcp-nb03 | category: RemoteTool | auth: ProjectManagedIdentity
SDK get           : maple-kb-mcp-nb03 RemoteTool | target ok: True


## 7. Register an agent version with an MCP tool, and call it
`AIProjectClient.agents.create_version` stores a **prompt agent**: model + instructions + tools, immutable per version (calling it again creates version 2).
The tool is an `MCPTool` pointing at the KB MCP URL via our project connection `maple-kb-mcp-nb03`. Other tool types you can put in `tools=[...]`: `FunctionTool`, `AzureAISearchTool`, `FileSearchTool`, `WebSearchTool`, `OpenApiTool`... (notebook 02 shows function and MCP tools).
Then we *call* the registered agent through the project's OpenAI-compatible Responses client by reference: `extra_body={"agent_reference": ...}`.

In [12]:
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

AGENT = "maple-policy-iq"
mcp_url = MCP_URL
instructions = (
    "You are Maple, the underwriting-policy assistant for Northwind Home Lending (fictional). "
    "ALWAYS call knowledge_base_retrieve before answering a policy question. Answer only from the retrieved passages, "
    "and cite them as 【policy id, section】. If the passages do not answer the question, reply exactly: "
    "\"I don't know - escalating to a human underwriter.\" Never promise a rate or approval.")
agent = pc.agents.create_version(
    agent_name=AGENT,
    definition=PromptAgentDefinition(model=CHAT, instructions=instructions, tools=[
        MCPTool(server_label="knowledge-base", server_url=mcp_url, require_approval="never",
                allowed_tools=["knowledge_base_retrieve"], project_connection_id=NB_CONN)]))
print("agent:", agent.name, "version", agent.version)
print("tools :", [(t.type, t.as_dict().get("server_label"), t.as_dict().get("project_connection_id")) for t in agent.definition.tools])

agent: maple-policy-iq version 1
tools : [('mcp', 'knowledge-base', 'maple-kb-mcp-nb03')]


In [13]:
def ask(question, tries=5):
    """Call the agent; back off on 429 / transient tool errors (shared, low-quota sandbox deployment)."""
    for attempt in range(tries):
        try:
            r = oai.responses.create(input=question, extra_body={"agent_reference": {"name": AGENT, "type": "agent_reference"}})
            break
        except Exception as exc:
            if attempt == tries - 1:
                raise
            wait = 15 * (attempt + 1)
            print(f"  retry in {wait}s ({type(exc).__name__}: {str(exc)[:80]})")
            time.sleep(wait)
    tools = [i for i in r.output if i.type in ("mcp_call", "mcp_list_tools")]
    print("Q:", question)
    print("tool calls:", [(i.type, getattr(i, "name", None), getattr(i, "error", None)) for i in tools])
    print("A:", r.output_text, "\n")
    return r

r1 = ask("What documents does a self-employed applicant need, and can they be pre-qualified automatically?")
r2 = ask("What is the minimum down payment on a $600,000 home?")

Q: What documents does a self-employed applicant need, and can they be pre-qualified automatically?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: A self-employed applicant needs to provide the following documents:
- Two years of Notice of Assessment (NOA) and T1 General forms.
- Two years of financial statements if the business is incorporated, plus business registration.

Self-employed applicants also need a minimum of 2 years Notice of Assessment history and are always referred for manual review, meaning they cannot be pre-qualified automatically and must go through manual underwriting.

Therefore, self-employed applicants cannot be pre-qualified automatically and must be manually reviewed by an underwriter.

This is summarized in the policies as follows:
- Document requirements for self-employed: two years NOA and T1 General, two years financial statements (if incorporated), and business registration.
- Manual referral is mandatory fo

Q: What is the minimum down payment on a $600,000 home?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: The minimum down payment on a $600,000 home is calculated as 5% of the first $500,000 plus 10% of the amount over $500,000. So for a $600,000 home, that would be:

- 5% of $500,000 = $25,000
- 10% of $100,000 (the amount over $500,000) = $10,000

Total minimum down payment = $25,000 + $10,000 = $35,000.

Therefore, the minimum down payment on a $600,000 home is $35,000【4:1†NHL-POL-101 Mortgage Eligibility Policy / 3. Down payment】. 



### Out of scope: it should say it does not know

In [14]:
r3 = ask("What is Northwind's policy on cryptocurrency income and NFT collateral?")
r3b = ask("Can gift funds be used for the down payment?")   # not covered by the corpus either

Q: What is Northwind's policy on cryptocurrency income and NFT collateral?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: I don't know - escalating to a human underwriter. 



Q: Can gift funds be used for the down payment?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: The retrieved policy documents do not specifically mention whether gift funds can be used for the down payment. Therefore, I don't know – escalating to a human underwriter. 



### Show the citations
The retrieved passages are in the `mcp_call` output; the references name the policy documents.

In [15]:
def show_citations(resp):
    """Print policy id + section of every passage the KB returned for this response."""
    for item in resp.output:
        if item.type == "mcp_call" and getattr(item, "output", None):
            for d in json.loads(item.output).get("documents", []):
                c = json.loads(d["content"])
                print(f"  [{c['policy_id']} / {c['section']}]  ({c['source']})")
show_citations(r2)
for item in r2.output:
    if item.type == "message":
        for part in item.content:
            for a in getattr(part, "annotations", []) or []:
                print("annotation:", a.type, "->", str(getattr(a, "url", "")).split("/docs/")[-1].split("?")[0])

  [NHL-POL-101 / 2. Property types]  (01-eligibility.md)
  [NHL-POL-101 / 3. Down payment]  (01-eligibility.md)
  [NHL-POL-103 / 1. All applicants]  (03-document-checklist.md)
annotation: url_citation -> 01-eligibility-3


## 8. Alternative: build the agent in code with Agent Framework `AzureAISearchContextProvider` [preview package]
This agent is **not registered** with Foundry: it exists in your Python process. Foundry only provides the model (Responses API) and the Search service provides retrieval. Steps:
1. **Client**: `FoundryChatClient(project_endpoint, model, credential)` binds to the project and the chat deployment.
2. **Context provider**: `AzureAISearchContextProvider(mode="agentic", knowledge_base_name=...)` runs the knowledge base *before* each model call and injects the passages.
3. **Agent**: `Agent(client=..., name=..., instructions=..., context_providers=[...], tools=[...])`.
4. **Run**: `await agent.run(text, session=...)` returns an `AgentResponse`; `agent.run(text, stream=True)` returns a `ResponseStream` of updates.

In [16]:
from agent_framework import Agent
from agent_framework.azure import AzureAISearchContextProvider
from agent_framework.foundry import FoundryChatClient

chat_client = FoundryChatClient(project_endpoint=ENDPOINT, model=CHAT, credential=credential)       # 1. client
provider = AzureAISearchContextProvider(                                                               # 2. context provider
    endpoint=SEARCH, credential=credential, mode="agentic", knowledge_base_name=KB,
    azure_openai_resource_url=OPENAI_URL, model=CHAT)
af_agent = Agent(client=chat_client, name="maple-af-iq",                                              # 3. agent
                 instructions="Answer only from the provided context and name the policy id. If unsure say you do not know.",
                 context_providers=[provider])
session = af_agent.create_session()                                                                    # conversation state
res = await af_agent.run("What is the maximum total debt service ratio?", session=session)             # 4. run
print(res.text)

The maximum total debt service (TDS) ratio is 44%.

Policy ID: NHL-POL-102


### Stream the answer
`stream=True` yields updates as the model produces tokens; `get_final_response()` gives the assembled `AgentResponse` (with usage).

In [17]:
stream = af_agent.run("And what is the maximum loan-to-value ratio?", session=session, stream=True)
seen = ""
async for update in stream:
    if update.text and update.text != seen:      # the last update repeats the whole text: skip it
        print(update.text, end="", flush=True); seen += update.text
final = await stream.get_final_response()
print("\n[usage]", {k: v for k, v in dict(final.usage_details or {}).items() if k.endswith("_token_count")})

The

 maximum

 loan

-to

-value

 (

... (10 outputs left out)

-POL

-102


[usage]

 {'input_token_count': 1098, 'output_token_count': 23, 'total_token_count': 1121, 'cache_creation_input_token_count': 0, 'cache_read_input_token_count': 0, 'reasoning_output_token_count': 0}


## 9. List and get everything back; where to find it in the portal
Registration is only useful if you can find the asset again. Read each one back with the SDK (the same calls you would use in a deployment script).
In the **Foundry portal** (ai.azure.com, your project `maple-mortgage`): **Build > Agents** shows `maple-policy-iq` and its versions; **Build > Knowledge** shows the project index `maple-policies`; **Management center > Connected resources** shows `maple-kb-mcp-nb03`. The knowledge source and knowledge base appear in the **Azure portal** under your Search service > *Agentic retrieval*.

In [18]:
ledger()
print()
ag = pc.agents.get(AGENT)
print("agent get      :", ag.name, "latest version", ag.versions.latest.version if hasattr(ag, "versions") else "?")
print("agent versions :", [v.version for v in pc.agents.list_versions(AGENT)])
print("connection get :", pc.connections.get(NB_CONN).name)
print("project index  :", [(i.name, i.version) for i in pc.indexes.list() if i.name == "maple-policies"])

project connections : ['appinsights', 'maple-kb-mcp-nb03', 'northwind-kb-mcp', 'search']


project indexes     : [('maple-policies', '1')]


agents              : ['maple-hosted', 'maple-learn-mcp', 'maple-policy-iq', 'maple-prompt', 'nb02-maple-fn', 'nb02-maple-learn-mcp']
search indexes      : ['northwind-policies']


knowledge sources   : ['northwind-policies-source']


knowledge bases     : ['northwind-kb']

agent get      : maple-policy-iq latest version 1


agent versions : ['1']
connection get : maple-kb-mcp-nb03


project index  : [('maple-policies', '1')]


## 10. Clean up
Delete what this notebook created for itself: the agent (all versions), the connection, and the project index asset.
The **Search index, knowledge source and knowledge base are shared** with notebooks 02, 06 and 07, so they stay unless you set `TEARDOWN_SHARED = True`. (`scripts/teardown` removes the whole resource group at the end of the workshop.)

In [19]:
TEARDOWN_SHARED = False      # True also removes the Search index + knowledge base/source (re-run this notebook to recreate them)

pc.agents.delete(AGENT)
print("deleted agent:", AGENT)
r = requests.delete(f"{ARM_CONN}/{NB_CONN}?api-version=2025-10-01-preview", headers=arm_headers, timeout=60)
print("deleted connection:", NB_CONN, "HTTP", r.status_code)
pc.indexes.delete(name="maple-policies", version="1")
print("deleted project index: maple-policies v1")
if TEARDOWN_SHARED:
    index_client.delete_knowledge_base(KB); index_client.delete_knowledge_source(KS); index_client.delete_index(INDEX)
    print("deleted shared Search assets")
print()
ledger()

deleted agent: maple-policy-iq


deleted connection: maple-kb-mcp-nb03 HTTP 200


deleted project index: maple-policies v1

project connections : ['appinsights', 'northwind-kb-mcp', 'search']


project indexes     : []


agents              : ['maple-hosted', 'maple-learn-mcp', 'maple-prompt', 'nb02-maple-fn', 'nb02-maple-learn-mcp']
search indexes      : ['northwind-policies']


knowledge sources   : ['northwind-policies-source']
knowledge bases     : ['northwind-kb']


## Key takeaways
- Index = your data; knowledge source = a pointer; knowledge base = planning + retrieval + references, reusable by many agents.
- Classic RAG (section 4): you control the query, cheap and predictable. Agentic retrieval (sections 5-7): better for multi-part, conversational questions; costs extra LLM calls.
- Chunk by section and keep the policy id in the chunk: citations become auditable.
- Instruct the agent to refuse when retrieval is empty. For FSI, an "I don't know, escalating" path beats a confident guess.
- Auth is Entra / managed identity throughout; no keys. Search needs `Search Index Data Reader` for the project identity.

**Next up:** [04 - Memory](04-memory.ipynb)